In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [2]:
# Estados computacionais
zero = np.array([1, 0], dtype=complex)
one  = np.array([0, 1], dtype=complex)


def ketbra(psi):
    """
    Constrói o operador |psi><psi|.
    """
    return np.outer(psi, psi.conj())


# Estados de Bell
phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2)

rho_phi_plus = ketbra(phi_plus)

rho_phi_plus

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0.5+0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [3]:
def verifica_densidade(rho, tol=1e-10):
    """
    Verifica as condições básicas para uma matriz densidade:
    
    rho = rho^\dagger
    rho >= 0
    Tr(rho) = 1
    """
    
    hermitiana = np.allclose(rho, rho.conj().T, atol=tol)
    autovalores = np.linalg.eigvalsh(rho)
    positiva = np.all(autovalores >= -tol)
    normalizada = np.isclose(np.trace(rho), 1, atol=tol)
    
    return {
        "hermitiana": hermitiana,
        "positiva": positiva,
        "traco_1": normalizada,
        "autovalores": autovalores
    }


verifica_densidade(rho_phi_plus)

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_50308/1935040699.py:5: SyntaxWarning: invalid escape sequence '\d'
  rho = rho^\dagger


{'hermitiana': True,
 'positiva': np.True_,
 'traco_1': np.True_,
 'autovalores': array([0., 0., 0., 1.])}

In [4]:
def tensor(*args):
    """
    Produto tensorial de vários operadores/estados.
    """
    resultado = args[0]
    
    for estado in args[1:]:
        resultado = np.kron(resultado, estado)
        
    return resultado

In [5]:
psi_000 = tensor(zero, zero, zero)

psi_000

array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])

In [6]:
def ghz(theta=0):
    """
    Estado GHZ de três qubits:
    
    |GHZ_theta> = (|000> + exp(i theta)|111>) / sqrt(2)
    """
    
    estado = (
        tensor(zero, zero, zero)
        + np.exp(1j * theta) * tensor(one, one, one)
    ) / np.sqrt(2)
    
    return estado


psi_ghz = ghz(theta=0)

rho_ghz = ketbra(psi_ghz)

verifica_densidade(rho_ghz)

{'hermitiana': True,
 'positiva': np.True_,
 'traco_1': np.True_,
 'autovalores': array([0., 0., 0., 0., 0., 0., 0., 1.])}

In [ ]:
def partial_trace(rho, trace_out, dims=None):
    """
    Calcula o traço parcial sobre os subsistemas indicados.

    Parâmetros
    ----------
    rho : ndarray
        Matriz densidade do sistema completo.

    trace_out : list
        Índices dos subsistemas sobre os quais realizar
        o traço parcial.

        Exemplo para ABC:
            [2]    -> Tr_C(rho) = rho_AB
            [1]    -> Tr_B(rho) = rho_AC
            [1, 2] -> Tr_BC(rho) = rho_A

    dims : list
        Dimensão de cada subsistema.
        Por padrão, todos são qubits.
    """

    if dims is None:
        n = int(np.log2(rho.shape[0]))
        dims = [2] * n

    n = len(dims)

    trace_out = sorted(trace_out, reverse=True)

    tensor_rho = rho.reshape(dims + dims)

    for i in trace_out:
        tensor_rho = np.trace(
            tensor_rho,
            axis1=i,
            axis2=i + len(dims)
        )

        dims.pop(i)

    dim_final = int(np.prod(dims))

    return tensor_rho.reshape(dim_final, dim_final)

In [8]:
rho_AB = partial_trace(rho_ghz, keep=[0, 1])

rho_AB

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [9]:
rho_AC = partial_trace(rho_ghz, keep=[0, 2])

rho_AC

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [10]:
rho_BC = partial_trace(rho_ghz, keep=[1, 2])

rho_BC

array([[0.5+0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
       [0. +0.j, 0. +0.j, 0. +0.j, 0.5+0.j]])

In [11]:
print("AB = AC:", np.allclose(rho_AB, rho_AC))
print("AB = BC:", np.allclose(rho_AB, rho_BC))

AB = AC: True
AB = BC: True


In [12]:
rho_A = partial_trace(rho_ghz, keep=[0])
rho_B = partial_trace(rho_ghz, keep=[1])
rho_C = partial_trace(rho_ghz, keep=[2])

print("rho_A =")
print(rho_A)

print("\nrho_B =")
print(rho_B)

print("\nrho_C =")
print(rho_C)

rho_A =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

rho_C =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


In [13]:
def espectro(rho):
    """
    Retorna os autovalores de uma matriz Hermitiana
    em ordem crescente.
    """
    return np.linalg.eigvalsh(rho)

In [14]:
print("Espectro de rho_A:")
print(espectro(rho_A))

print("\nEspectro de rho_AB:")
print(espectro(rho_AB))

Espectro de rho_A:
[0.5 0.5]

Espectro de rho_AB:
[0.  0.  0.5 0.5]


In [16]:
def verifica_overlap(rho_S, subsistemas_S,
                     rho_T, subsistemas_T):
    """
    Verifica a consistência das marginais no subsistema
    compartilhado.

    subsistemas_S e subsistemas_T são listas com os índices
    dos qubits presentes em cada marginal.
    """

    overlap = sorted(
        set(subsistemas_S).intersection(subsistemas_T)
    )

    if len(overlap) == 0:
        raise ValueError("As marginais não possuem overlap.")

    # posições relativas dentro de cada marginal
    keep_S = [
        subsistemas_S.index(i)
        for i in overlap
    ]

    keep_T = [
        subsistemas_T.index(i)
        for i in overlap
    ]

    reduzida_S = partial_trace(
        rho_S,
        keep=keep_S,
        dims=[2] * len(subsistemas_S)
    )

    reduzida_T = partial_trace(
        rho_T,
        keep=keep_T,
        dims=[2] * len(subsistemas_T)
    )

    return np.allclose(reduzida_S, reduzida_T)

In [17]:
verifica_overlap(
    rho_AB, [0, 1],
    rho_AC, [0, 2]
)

True

In [18]:
rho_AB_bell = rho_phi_plus.copy()
rho_AC_bell = rho_phi_plus.copy()

print(
    verifica_overlap(
        rho_AB_bell, [0, 1],
        rho_AC_bell, [0, 2]
    )
)

True
